# Spark Exercises 09 — Capstone: Streamline Analytics

You are a data engineer at **Streamline**, an online store. Two worlds of data have landed: the **transactional** tables (orders, customers, products, regions) and the **behavioural** clickstream (`events.jsonl`). Your mission: build an end-to-end analytics pipeline that a business team could actually use — clean, enrich, rank, pivot, fuse transactions with behaviour into a *customer-360* table, and write the result. This capstone uses everything from chapters 1–8.

**Data:** `orders.csv`, `customers.csv`, `products.csv`, `regions.csv`, `events.jsonl`

---

> **Solutions version** — try it yourself first from `Exercises.ipynb`.

### 1. **Setup** — open a local `SparkSession` (already written for you). In Foundry the session is created for you; locally we open one ourselves. `F` is the functions module — almost every column expression lives there.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("spark-course")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Spark 3.5.1


### 2. Read all five sources: `orders`, `customers`, `products`, `regions` (CSV, inferSchema) and `events` (JSON).

In [2]:
orders = spark.read.csv("data/orders.csv", header=True, inferSchema=True)
customers = spark.read.csv("data/customers.csv", header=True, inferSchema=True)
products = spark.read.csv("data/products.csv", header=True, inferSchema=True)
regions = spark.read.csv("data/regions.csv", header=True, inferSchema=True)
events = spark.read.json("data/events.jsonl")
print(orders.count(), customers.count(), products.count(), regions.count(), events.count())

8000 500 20 5 2500


### 3. **Clean the orders.** Keep only real sales (`status == 'completed'` and `quantity > 0`), and add `revenue = round(quantity * unit_price, 2)`. Call it `sales`. How many rows remain?

In [3]:
sales = (orders
         .filter((F.col("status") == "completed") & (F.col("quantity") > 0))
         .withColumn("revenue", F.round(F.col("quantity") * F.col("unit_price"), 2)))
sales.count()

6533

### 4. **Enrich.** Join `sales` to `customers`, then `regions` (on `region_id`), then `products` (on `product_sku`) into `fact`. Show `order_id, region_name, category, segment, revenue` for 5 rows.

In [4]:
fact = (sales
        .join(customers, "customer_id")
        .join(regions, "region_id")
        .join(products, "product_sku"))
fact.select("order_id", "region_name", "category", "segment", "revenue").show(5)

+----------+-------------+----------+--------+-------+
|  order_id|  region_name|  category| segment|revenue|
+----------+-------------+----------+--------+-------+
|ORD-000001| Asia Pacific|     books|Consumer|  325.8|
|ORD-000002|North America|stationery|Consumer|  31.46|
|ORD-000003|       Europe|stationery|     SMB|  10.26|
|ORD-000004|North America|      home|Consumer| 149.52|
|ORD-000005|       Europe|      home|Consumer| 187.91|
+----------+-------------+----------+--------+-------+
only showing top 5 rows



### 5. **KPI 1 — revenue by region.** Total revenue per `region_name`, highest first.

In [5]:
fact.groupBy("region_name") \
    .agg(F.round(F.sum("revenue"), 2).alias("revenue")) \
    .orderBy(F.col("revenue").desc()).show()

+--------------------+---------+
|         region_name|  revenue|
+--------------------+---------+
|       North America|627474.99|
|              Europe|361471.58|
|        Asia Pacific|345334.25|
|       Latin America|155806.62|
|Middle East & Africa| 71673.84|
+--------------------+---------+



### 6. **KPI 2 — a region × category matrix.** Use `pivot` to get total revenue with `region_name` as rows and `category` as columns.

In [6]:
fact.groupBy("region_name").pivot("category") \
    .agg(F.round(F.sum("revenue"), 0)) \
    .orderBy("region_name").show()

+--------------------+--------+-------+-----------+-------+----------+
|         region_name| apparel|  books|electronics|   home|stationery|
+--------------------+--------+-------+-----------+-------+----------+
|        Asia Pacific| 61017.0|53234.0|   159709.0|52706.0|   18669.0|
|              Europe| 81358.0|54648.0|   147753.0|56446.0|   21267.0|
|       Latin America| 26256.0|24874.0|    75022.0|21168.0|    8486.0|
|Middle East & Africa| 15264.0|12305.0|    33589.0| 6884.0|    3633.0|
|       North America|123248.0|99386.0|   286424.0|86204.0|   32214.0|
+--------------------+--------+-------+-----------+-------+----------+



### 7. **KPI 3 — top 3 products per region.** Sum revenue per (`region_name`, `product_name`), then use a window ranked by revenue within each region and keep the top 3.

In [7]:
from pyspark.sql.window import Window

prod_rev = fact.groupBy("region_name", "product_name") \
               .agg(F.round(F.sum("revenue"), 2).alias("revenue"))
w = Window.partitionBy("region_name").orderBy(F.col("revenue").desc())
prod_rev.withColumn("rnk", F.row_number().over(w)) \
        .filter(F.col("rnk") <= 3) \
        .orderBy("region_name", "rnk").show(15)

+--------------------+--------------------+---------+---+
|         region_name|        product_name|  revenue|rnk|
+--------------------+--------------------+---------+---+
|        Asia Pacific|Noise-Cancel Head...| 95140.08|  1|
|        Asia Pacific|   Bluetooth Speaker| 36227.69|  2|
|        Asia Pacific|              Hoodie| 29599.46|  3|
|              Europe|Noise-Cancel Head...| 89426.67|  1|
|              Europe|              Hoodie| 42250.43|  2|
|              Europe|   Bluetooth Speaker|  37813.5|  3|
|       Latin America|Noise-Cancel Head...| 45440.69|  1|
|       Latin America|   Bluetooth Speaker| 17854.63|  2|
|       Latin America|Data Engineering 101|  14175.3|  3|
|Middle East & Africa|Noise-Cancel Head...| 24894.46|  1|
|Middle East & Africa|              Hoodie|  8505.76|  2|
|Middle East & Africa|Data Engineering 101|  7590.48|  3|
|       North America|Noise-Cancel Head...|173610.85|  1|
|       North America|   Bluetooth Speaker| 72386.79|  2|
|       North 

### 8. **Customer value.** Per customer, compute `total_spent`, `n_orders`, and `last_order` (max `order_ts`). Call it `cust_value`. Show the top 5 spenders.

In [8]:
cust_value = sales.groupBy("customer_id").agg(
    F.round(F.sum("revenue"), 2).alias("total_spent"),
    F.count("*").alias("n_orders"),
    F.max("order_ts").alias("last_order"),
)
cust_value.orderBy(F.col("total_spent").desc()).show(5)

+-----------+-----------+--------+-------------------+
|customer_id|total_spent|n_orders|         last_order|
+-----------+-----------+--------+-------------------+
|  CUST-0098|    8890.27|      13|2024-11-26 00:51:00|
|  CUST-0393|     8841.7|      13|2024-11-21 18:00:00|
|  CUST-0287|    8796.69|      18|2024-12-26 03:12:00|
|  CUST-0115|    8318.07|      20|2024-12-18 18:08:00|
|  CUST-0127|    8126.33|      16|2024-12-12 18:39:00|
+-----------+-----------+--------+-------------------+
only showing top 5 rows



### 9. **Rank customers into spend deciles.** Add `decile = ntile(10)` over a window ordered by `total_spent` descending (decile 1 = top spenders). Show 10 rows.

In [9]:
w_spend = Window.orderBy(F.col("total_spent").desc())
cust_ranked = cust_value.withColumn("decile", F.ntile(10).over(w_spend))
cust_ranked.select("customer_id", "total_spent", "decile").show(10)

+-----------+-----------+------+
|customer_id|total_spent|decile|
+-----------+-----------+------+
|  CUST-0098|    8890.27|     1|
|  CUST-0393|     8841.7|     1|
|  CUST-0287|    8796.69|     1|
|  CUST-0115|    8318.07|     1|
|  CUST-0127|    8126.33|     1|
|  CUST-0109|    8014.13|     1|
|  CUST-0361|    7959.27|     1|
|  CUST-0298|    7905.64|     1|
|  CUST-0237|    7617.04|     1|
|  CUST-0036|    7537.08|     1|
+-----------+-----------+------+
only showing top 10 rows



### 10. **Behavioural side — the funnel.** From `events`, count events per `event_type`, ordered most to least (the classic page_view → search → add_to_cart → purchase funnel).

In [10]:
events.groupBy("event_type").count().orderBy(F.col("count").desc()).show()

+-----------+-----+
| event_type|count|
+-----------+-----+
|  page_view| 1246|
|     search|  541|
|add_to_cart|  501|
|   purchase|  212|
+-----------+-----+



### 11. **Purchases by device.** Explode the `items` array of `purchase` events and compute purchase revenue (`qty * price`) per `device.os`.

In [11]:
purchase_lines = (events
    .filter(F.col("event_type") == "purchase")
    .select("device.os", F.explode("items").alias("item")))
purchase_lines.groupBy("os") \
    .agg(F.round(F.sum(F.col("item.qty") * F.col("item.price")), 2).alias("revenue")) \
    .orderBy(F.col("revenue").desc()).show()

+-------+--------+
|     os| revenue|
+-------+--------+
|Android|15693.93|
|  macOS|14897.16|
|  Linux|13613.95|
|Windows|12048.69|
|    iOS| 9926.78|
+-------+--------+



### 12. **Fuse the two worlds — customer-360.** Build per-user event counts from `events` (rename `user_id` → `customer_id`), then **left join** `customers` → `cust_value` → event counts so every customer appears even with no orders/events. Fill missing numbers with 0. Show 10 rows of `customer_id, segment, total_spent, n_orders, n_events`.

In [12]:
event_counts = events.groupBy(F.col("user_id").alias("customer_id")) \
                      .agg(F.count("*").alias("n_events"))
c360 = (customers
        .join(cust_value, "customer_id", "left")
        .join(event_counts, "customer_id", "left")
        .fillna({"total_spent": 0, "n_orders": 0, "n_events": 0}))
c360.select("customer_id", "segment", "total_spent", "n_orders", "n_events").show(10)

+-----------+----------+-----------+--------+--------+
|customer_id|   segment|total_spent|n_orders|n_events|
+-----------+----------+-----------+--------+--------+
|  CUST-0001|       SMB|    1191.24|       8|       4|
|  CUST-0002|Enterprise|     3486.2|      15|       9|
|  CUST-0003|Enterprise|    3299.85|      19|       5|
|  CUST-0004|  Consumer|    4020.27|      10|       5|
|  CUST-0005|  Consumer|    2604.73|      12|       3|
|  CUST-0006|       SMB|    3887.71|      12|       3|
|  CUST-0007|  Consumer|    3883.06|      10|       9|
|  CUST-0008|       SMB|    3776.13|      15|       2|
|  CUST-0009|  Consumer|    3182.92|      12|       6|
|  CUST-0010|  Consumer|    2019.83|      14|       6|
+-----------+----------+-----------+--------+--------+
only showing top 10 rows



### 13. **A KPI in pure SQL.** Register `c360` as a temp view and, with `spark.sql`, return average `total_spent` and average `n_events` **per `segment`**.

In [13]:
c360.createOrReplaceTempView("c360")
spark.sql("""
    SELECT segment,
           round(avg(total_spent), 2) AS avg_spent,
           round(avg(n_events), 1)   AS avg_events
    FROM c360
    GROUP BY segment
    ORDER BY avg_spent DESC
""").show()

+----------+---------+----------+
|   segment|avg_spent|avg_events|
+----------+---------+----------+
|       SMB|  3218.26|       4.8|
|  Consumer|  3178.41|       5.2|
|Enterprise|   2791.3|       4.9|
+----------+---------+----------+



### 14. **Ship it.** Write the enriched `fact` table as Parquet, partitioned by `region_name`, into a temp folder. List the partition directories to confirm.

In [14]:
import tempfile, os
OUT = tempfile.mkdtemp()
path = os.path.join(OUT, "fact")
fact.write.mode("overwrite").partitionBy("region_name").parquet(path)
print(sorted(os.listdir(path)))

['._SUCCESS.crc', '_SUCCESS', 'region_name=Asia Pacific', 'region_name=Europe', 'region_name=Latin America', 'region_name=Middle East & Africa', 'region_name=North America']
